In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
from time import time
import matplotlib.pyplot as plt
from main import ModelClass

from scipy.optimize import minimize

model = ModelClass()

par = model.par
sol = model.sol
sim = model.sim

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This model is used to find a certain condition. Currently, I just minimize the change in average wage level, because it was not easy to get by just setting parameters.

In [ ]:
param_spec = {
    "A":        {"value": 1.0,  "bounds": (0.5, 5.0),  "active": True},
    "N_y":      {"value": 20,   "bounds": (1, 100),    "active": True},
    "alpha":    {"value": 0.5,  "bounds": (0.01, 0.99),  "active": True},
    "theta_l_y":{"value": 1.0,  "bounds": (0.1, 5.0),  "active": True},
    "theta_h_y":{"value": 3.6,  "bounds": (0.1, 5.0),  "active": True},
    "theta_l_o":{"value": 1.1,  "bounds": (0.1, 5.0),  "active": True},
    "theta_h_o":{"value": 3.2,  "bounds": (0.1, 5.0),  "active": True},
    "mu_y":     {"value": 2.7,  "bounds": (0.1, 10.0), "active": True},
    "mu_o":     {"value": 1.2,  "bounds": (0.1, 10.0), "active": False},
    "rho_h":    {"value": 0.2,  "bounds": (0.1, 0.99), "active": False},
    "rho_l":    {"value": 0.2,  "bounds": (0.1, 0.99), "active": True},
    "c":        {"value": 0.7,  "bounds": (0.1, 10.0),  "active": True},
}


active_names = [k for k, v in param_spec.items() if v["active"]]
x0 = np.array([param_spec[k]["value"] for k in active_names], dtype=float)
bounds = [param_spec[k]["bounds"] for k in active_names]

x0_random = np.array([np.random.uniform(low, high) for (low, high) in bounds], dtype=float)

def apply_params(par, names, values):
    for name, val in zip(names, values):
        setattr(par, name, float(val))
    return par

def objective(x):
    # Initialize all non-active parameters to their default values
    for name, spec in param_spec.items():
        if not spec["active"]:
            setattr(par, name, spec["value"])

    apply_params(par, active_names, x)

    try:
        model.solve()

        parameter_names = ["rho_h"]
        parameter_values = [0.99]

        model.simulate(parameter_names, parameter_values)

        return model.sim.avg_wage[10] - model.sim.avg_wage[11]
    
    except Exception:
        # penalty if solver fails
        return 1e12

result = minimize(
    objective,
    x0=x0_random,
    bounds=bounds,
    method="nelder-mead",
)

# --- write back optimized values ---
optimized_values = result.x
apply_params(par, active_names, optimized_values)

print("Success:", result.success)
print("Message:", result.message)
print("Objective value:", result.fun)
print("Optimized parameters:")
for name in active_names:
    print("par." + name + " = ", getattr(par, name))

# final solve/sim with optimal values
model.solve()
model.simulate(active_names, list(optimized_values))

Success: True
Message: Optimization terminated successfully.
Objective value: -12.87115678537874
Optimized parameters:
par.A =  4.832607922589209
par.N_y =  1.0000011534551123
par.alpha =  0.9572581828189745
par.theta_l_y =  1.2499583697315901
par.theta_h_y =  4.9999743090939095
par.theta_l_o =  4.167447833189932
par.theta_h_o =  0.10005280243991468
par.mu_y =  3.8030080440648364
par.rho_l =  0.3493005835288795
par.c =  0.8492451725763352
Return condition: -12.87115678537874
